In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurações visuais do Seaborn
sns.set_theme(style="whitegrid")

# 1. Definir os caminhos
BASE_DIR = Path("/home/students/moliveira/telegram-political-toxicity/reports/topic_modeling")

# Lista para armazenar o resumo de todos os modelos
resumo_modelos = []
# Dicionário para armazenar o DataFrame completo de cada modelo
dados_modelos = {}

print("Procurando e carregando arquivos topic_info.json...\n")

# 2. Varrer todas as subpastas buscando o topic_info.json
for json_path in BASE_DIR.rglob("topic_info.json"):
    # Extrai o nome da métrica otimizada (ex: best_coherence_cv)
    nome_modelo = json_path.parent.parent.name
    
    # Carrega o JSON
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    df = pd.DataFrame(data)
    dados_modelos[nome_modelo] = df
    
    # 3. Calcular métricas do modelo
    total_mensagens = df['Count'].sum()
    
    # Busca a contagem de outliers (Tópico -1), se existir
    outliers_row = df[df['Topic'] == -1]
    total_outliers = outliers_row['Count'].values[0] if not outliers_row.empty else 0
    perc_outliers = (total_outliers / total_mensagens) * 100
    
    # Quantidade de tópicos válidos (excluindo o -1)
    topicos_validos = len(df[df['Topic'] != -1])
    
    # Quantidade de mensagens no maior tópico válido (geralmente o 0)
    top_1_valido = df[df['Topic'] != -1]['Count'].max()
    perc_top_1 = (top_1_valido / total_mensagens) * 100
    
    # Adiciona ao resumo
    resumo_modelos.append({
        'Configuração': nome_modelo,
        'Qtd. Tópicos Válidos': topicos_validos,
        '% Outliers (Tópico -1)': perc_outliers,
        '% Mensagens no Maior Tópico (Tópico 0)': perc_top_1,
        'Total Mensagens Processadas': total_mensagens
    })

# 4. Criar e exibir a Tabela de Resumo Comparativa
df_resumo = pd.DataFrame(resumo_modelos).sort_values(by='Qtd. Tópicos Válidos')
print("=== RESUMO COMPARATIVO DOS MODELOS ===")
display(df_resumo.round(2))

# ---------------------------------------------------------
# 5. Plotar a Distribuição de Outliers vs Tópicos Válidos
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_resumo, 
    x='% Outliers (Tópico -1)', 
    y='Configuração', 
    hue='Configuração',
    palette='Reds_r',
    legend=False
)
plt.title('Percentual de Outliers (Mensagens Não Classificadas) por Configuração', fontsize=14)
plt.xlabel('% de Outliers')
plt.ylabel('Modelo')
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 6. Plotar a Distribuição dos Top 5 Tópicos (Excluindo Outliers)
# ---------------------------------------------------------
for nome, df in dados_modelos.items():
    # Ignora o tópico -1 para o gráfico de barras
    df_validos = df[df['Topic'] != -1].copy()
    # Pega os 5 maiores
    top_5 = df_validos.nlargest(5, 'Count')
    
    plt.figure(figsize=(10, 4))
    sns.barplot(
        data=top_5, 
        x='Count', 
        y='Name', 
        hue='Name',
        palette='viridis',
        legend=False
    )
    plt.title(f'Top 5 Maiores Tópicos Válidos - {nome}', fontsize=12)
    plt.xlabel('Número de Mensagens')
    plt.ylabel('Tópico')
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# 7. Extrair e exibir as palavras-chave do Top 3 Tópicos de cada modelo
# ---------------------------------------------------------
print("\n=== REPRESENTAÇÃO DOS TOP 3 TÓPICOS POR MODELO ===")
for nome, df in dados_modelos.items():
    print(f"\n-> Modelo: {nome}")
    df_validos = df[df['Topic'] != -1].nlargest(3, 'Count')
    
    for _, row in df_validos.iterrows():
        palavras = ", ".join(row['Representation'][:7]) # Pega as 7 primeiras palavras
        print(f"   Tópico {row['Topic']} ({row['Count']} msgs): {palavras}")

In [ ]:
df = pd.read_parquet(f"{BASE_DIR}best_result/all-distilroberta-v1/post_topics.parquet")
df